In [4]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from mpl_toolkits.mplot3d import Axes3D
import seaborn as sns
from scipy import stats
from scipy.spatial.distance import pdist, squareform
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, SpectralClustering
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ============================================================
# Cell 2: Load Data
# ============================================================
# ---- EDIT THIS SECTION ----
# Option A: Load from .npy
rsa_data = np.load('searchlight_rsa.npy')  # (20, 575, 56949)

# Option B: If you have a searchlight center coordinates file (MNI)
# This is needed for brain mapping later
# center_coords = np.load('searchlight_centers.npy')  # (56949, 3) MNI x,y,z

# Option C: If you have a brain mask in nifti
# mask_img = nib.load('brain_mask.nii.gz')
# mask_data = mask_img.get_fdata().astype(bool)
# ----------------------------

n_subjects, n_timepoints, n_voxels = rsa_data.shape
print(f"Data shape: {rsa_data.shape}")
print(f"  Subjects:    {n_subjects}")
print(f"  Timepoints:  {n_timepoints}")
print(f"  Voxel centers: {n_voxels}")
print(f"  Data range:  [{np.nanmin(rsa_data):.4f}, {np.nanmax(rsa_data):.4f}]")
print(f"  NaN count:   {np.isnan(rsa_data).sum()}")
print(f"  Mean (overall): {np.nanmean(rsa_data):.6f}")

Data shape: (20, 575, 56949)
  Subjects:    20
  Timepoints:  575
  Voxel centers: 56949
  Data range:  [-0.3053, 0.5207]
  NaN count:   0
  Mean (overall): 0.009930


In [5]:
# ============================================================
# Cell 3: Data Preprocessing
# ============================================================
# Handle NaNs — replace with 0 (no coupling)
rsa_clean = np.nan_to_num(rsa_data, nan=0.0)

# Group average
group_data = rsa_clean.mean(axis=0)  # (575, 56949)
print(f"Group average shape: {group_data.shape}")

# Also keep individual subject data for later robustness checks
# Z-score across timepoints for each voxel (per subject)
group_scaled = StandardScaler().fit_transform(group_data)  # z-score each voxel across time
print(f"Scaled data shape: {group_scaled.shape}")
print(f"Scaled data range: [{group_scaled.min():.2f}, {group_scaled.max():.2f}]")

Group average shape: (575, 56949)
Scaled data shape: (575, 56949)
Scaled data range: [-5.96, 5.05]


In [10]:
tmp_image_path = r'N:\Experimental_Data\yujunchen\projects/IAPS_fMRI_RSA\fMRI_singletrial_betas\nifti\Nt1.img'
# load one image to get the dimensions and make the mask

tmp_img = nib.load(tmp_image_path)

In [13]:
import numpy as np
import nibabel as nib
import pandas as pd
from nilearn.image import new_img_like
import nilearn.image as nlimg

# Load data
rsa_data = np.load('searchlight_rsa.npy')  # (20, 575, 56949)

# 1. Average over subjects → (575, 56949)
mean_over_subjects = np.nanmean(rsa_data, axis=0)

# 2. Max over timepoints → (56949,)
max_over_time = np.nanmax(mean_over_subjects, axis=0)

# 3. Map back to 3D volume
voxel_center_id = pd.read_csv(
    r"N:\Experimental_Data\yujunchen\projects\IAPS_Searchlight\outputs\voxel_center_id.csv",
    header=0, index_col=0
)

x, y, z = 53, 63, 46
vol = np.full(x * y * z, np.nan)
vol[np.squeeze(np.array(voxel_center_id))] = max_over_time
vol_3d = vol.reshape([x, y, z])

# 4. Create nifti and save
tmp_img = nib.load(r'N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_RSA\fMRI_singletrial_betas\nifti\Nt1.img')  # same template you used before
result_img = new_img_like(tmp_img, vol_3d)

# Optional: cluster threshold
# result_img = nlimg.threshold_img(result_img, threshold=0, cluster_threshold=30)

nib.save(result_img, 'searchlight_rsa_max_over_time.nii.gz')
print(f"Saved. Non-NaN voxels: {np.sum(~np.isnan(vol_3d))}")
print(f"Value range: [{np.nanmin(max_over_time):.4f}, {np.nanmax(max_over_time):.4f}]")

Saved. Non-NaN voxels: 56949
Value range: [0.0127, 0.0898]


In [15]:
import numpy as np
import nibabel as nib
import pandas as pd
from nilearn.image import new_img_like
import nilearn.image as nlimg
import os

# Load data
rsa_data = np.load('searchlight_rsa.npy')  # (20, 575, 56949)

# Threshold parameter
threshold = 0.05

# 1. Average over subjects → (575, 56949)
mean_over_subjects = np.nanmean(rsa_data, axis=0)

# 2. Max over timepoints → (56949,)
max_over_time = np.nanmax(mean_over_subjects, axis=0)

# 3. Apply threshold — discard voxels below threshold
max_over_time[max_over_time < threshold] = np.nan

# 4. Map back to 3D volume
voxel_center_id = pd.read_csv(
    r"N:\Experimental_Data\yujunchen\projects\IAPS_Searchlight\outputs\voxel_center_id.csv",
    header=0, index_col=0
)

x, y, z = 53, 63, 46
vol = np.full(x * y * z, np.nan)
vol[np.squeeze(np.array(voxel_center_id))] = max_over_time
vol_3d = vol.reshape([x, y, z])

# 5. Create nifti and save to ROI folder
os.makedirs('ROI', exist_ok=True)
thre_str = f"thre_{str(threshold).replace('0.', '')}"  # e.g. 'thre_053'
out_path = os.path.join('ROI', f'searchlight_rsa_max_{thre_str}.nii.gz')

tmp_img = nib.load(r'N:\Experimental_Data\yujunchen\projects\IAPS_fMRI_RSA\fMRI_singletrial_betas\nifti\Nt1.img')
result_img = new_img_like(tmp_img, vol_3d)

nib.save(result_img, out_path)
print(f"Saved to {out_path}")
print(f"Non-NaN voxels: {np.sum(~np.isnan(vol_3d))}")
print(f"Value range: [{np.nanmin(max_over_time):.4f}, {np.nanmax(max_over_time):.4f}]")

Saved to ROI\searchlight_rsa_max_thre_05.nii.gz
Non-NaN voxels: 12124
Value range: [0.0500, 0.0898]
